In [7]:
import torchvision
# import argparse
import yaml
import os
from torchvision.utils import make_grid
from tqdm import tqdm
from models.unet_cond_base import Unet
from scheduler.linear_noise_scheduler import LinearNoiseScheduler
from utils.config_utils import *

config_path = 'config/icd2fc_image_cond.yaml'
# Read the config file #
with open(config_path, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)
print(config)
########################

diffusion_config = config['diffusion_params']
dataset_config = config['dataset_params']
# dataset_config['im_size'] = 120
diffusion_model_config = config['ldm_params']
autoencoder_model_config = config['autoencoder_params']
train_config = config['train_params']
diffusion_config['num_timesteps'] = 100

########## Create the noise scheduler #############
scheduler = LinearNoiseScheduler(num_timesteps=diffusion_config['num_timesteps'],
                                 beta_start=diffusion_config['beta_start'],
                                 beta_end=diffusion_config['beta_end'])
###############################################

text_tokenizer = None
text_model = None

############# Validate the config #################
condition_config = get_config_value(diffusion_model_config, key='condition_config', default_value=None)
assert condition_config is not None, ("This sampling script is for text conditional "
                                      "but no conditioning config found")
condition_types = get_config_value(condition_config, 'condition_types', [])
assert 'text' in condition_types, ("This sampling script is for text conditional "
                                    "but no text condition found in config")
validate_text_config(condition_config)
###############################################


{'dataset_params': {'im_path': 'placeholder', 'im_channels': 1, 'im_size': 116, 'name': 'ICDFC'}, 'diffusion_params': {'num_timesteps': 1000, 'beta_start': 0.00085, 'beta_end': 0.012}, 'ldm_params': {'down_channels': [128, 256, 384, 512], 'mid_channels': [512, 384], 'down_sample': [True, True, True], 'attn_down': [True, True, True], 'time_emb_dim': 512, 'norm_channels': 32, 'num_heads': 8, 'conv_out_channels': 128, 'num_down_layers': 1, 'num_mid_layers': 2, 'num_up_layers': 1, 'condition_config': {'condition_types': ['text'], 'text_condition_config': {'text_embed_model': 'clip', 'train_text_embed_model': False, 'text_embed_dim': 512, 'cond_drop_prob': 0.1}, 'image_condition_config': {'image_condition_input_channels': 1, 'image_condition_output_channels': 1, 'image_condition_h': 120, 'image_condition_w': 120, 'cond_drop_prob': 0.1}}}, 'autoencoder_params': {'z_channels': 1, 'codebook_size': 8192, 'down_channels': [64, 128, 256, 256], 'mid_channels': [256, 256], 'down_sample': [True, Tru

In [8]:
from models.cholesky_ddpm import LatentMLPDiffusion
import torch
device = 'cuda:5'
model = LatentMLPDiffusion(model_config=diffusion_model_config, input_shape=[256+16,256]).to(device)
model.eval()
latent_tag = 'DualSPD_thr25'
model.load_state_dict(torch.load(f'icd2fc/best_{latent_tag}vqvae_ddpm_ckpt_text_image_cond_clip.pth'))
from models.vqvae import VQVAE, SPD_VQVAE
autoencoder_config = config['autoencoder_params']

model1 = SPD_VQVAE(input_dim=116,
                      model_config=autoencoder_config,
                   num_latents=256,).to(device)
model1.load_state_dict(torch.load('icd2fc/DualSPD_thr25_m1vqvae_autoencoder_ckpt.pth',
                                 map_location=device))

model2 = SPD_VQVAE(input_dim=116,
                  model_config=autoencoder_config,
                   num_latents=16, # 
                   codebook_size=16 # 
                  ).to(device)
model2.load_state_dict(torch.load('icd2fc/DualSPD_thr25_m2vqvae_autoencoder_ckpt.pth',
                                 map_location=device))
model1.eval()
model2.eval()

def reverse_decomp(comp1, comp2):
    n = comp1.shape[-1]
    batch_mat = torch.zeros(len(comp1), n, n)
    tril_idx = torch.tril_indices(n, n)
    triu_idx = torch.triu_indices(n, n)
    batch_mat[:, tril_idx[0], tril_idx[1]] = torch.fft.irfft(torch.fft.rfft(comp1[:, tril_idx[0], tril_idx[1]], dim=-1) + torch.fft.rfft(comp2[:, tril_idx[0], tril_idx[1]], dim=-1), n=tril_idx.shape[-1], dim=-1).detach().cpu()
    batch_mat[:, triu_idx[0], triu_idx[1]] = batch_mat[:, triu_idx[1], triu_idx[0]]
    return batch_mat
    

In [3]:
data = torch.load('ntp_text_embed_train.pth')

In [4]:
data['text_embed'].shape

torch.Size([43049, 300, 512])

In [5]:
all_text_embed = data['text_embed']

In [9]:
pred_zs = []
all_decoded_output1 = []
all_decoded_output2 = []
bsz = 512
startbatchi = len(all_text_embed)//4 + len(all_text_embed)//4 + len(all_text_embed)//4
endbatchi = len(all_text_embed)
for batchi in range(startbatchi, endbatchi, bsz):
    cond_input = {'text': all_text_embed[batchi:batchi+bsz]}
    ########### Sample random noise latent ##########
    # For not fixing generation with one sample
    xt = torch.randn((len(cond_input['text']),
                      256+16,
                      256)).to(device)
    ###############################################
    cond_input['text'] = cond_input['text'].to(device)

    ################# Sampling Loop ########################
    with torch.no_grad():
        for i in tqdm(reversed(range(diffusion_config['num_timesteps'])), desc=f'batch {batchi}'):
            # Get prediction of noise
            t = (torch.ones((xt.shape[0],)) * i).long().to(device)
            noise_pred_cond = model(xt, t, cond_input=cond_input)
            noise_pred = noise_pred_cond
            
            # Use scheduler to get x0 and xt-1
            xt, x0_pred = scheduler.sample_prev_timestep(xt, noise_pred, torch.as_tensor(i).to(device), spd=False)
            
            # Save x0
            if i == 0:
                # Decode ONLY the final image to save time
                ims = xt
            else:
                ims = x0_pred
            
            pred_z = ims

    pred_zs.append(pred_z.detach().cpu())
    with torch.no_grad():
        all_decoded_output1.append(model1.decode(pred_z[:, :256])[0].detach().cpu().squeeze())
        all_decoded_output2.append(model2.decode(pred_z[:, 256:])[0].detach().cpu().squeeze())
        

all_decoded_output1 = torch.cat(all_decoded_output1)
all_decoded_output2 = torch.cat(all_decoded_output2)
pred_zs = torch.cat(pred_zs)

batch 32286: 100it [06:23,  3.83s/it]
batch 32798: 100it [06:26,  3.87s/it]
batch 33310: 100it [06:32,  3.92s/it]
batch 33822: 100it [06:33,  3.94s/it]
batch 34334: 100it [06:33,  3.93s/it]
batch 34846: 100it [06:32,  3.92s/it]
batch 35358: 100it [06:32,  3.92s/it]
batch 35870: 100it [06:32,  3.93s/it]
batch 36382: 100it [06:32,  3.92s/it]
batch 36894: 100it [06:32,  3.92s/it]
batch 37406: 100it [06:32,  3.92s/it]
batch 37918: 100it [06:31,  3.92s/it]
batch 38430: 100it [06:32,  3.92s/it]
batch 38942: 100it [06:31,  3.92s/it]
batch 39454: 100it [06:32,  3.92s/it]
batch 39966: 100it [06:32,  3.92s/it]
batch 40478: 100it [06:31,  3.92s/it]
batch 40990: 100it [06:32,  3.92s/it]
batch 41502: 100it [06:32,  3.92s/it]
batch 42014: 100it [06:33,  3.93s/it]
batch 42526: 100it [06:33,  3.94s/it]
batch 43038: 100it [00:09, 10.42it/s]


In [10]:
pred_zs.shape

torch.Size([10763, 272, 256])

In [11]:
all_decoded_output = reverse_decomp(all_decoded_output1.squeeze(), all_decoded_output2.squeeze()).clamp(-1., 1.)
all_decoded_output[:, torch.arange(all_decoded_output.shape[-1]), torch.arange(all_decoded_output.shape[-1])] = 1
all_decoded_output.shape

torch.Size([10763, 116, 116])

In [12]:
torch.save(
    {'pred_zs': pred_zs.detach().cpu(), 
     'gen_img': all_decoded_output.detach().cpu(), 
    }, 'ntp_gen-fc_train3.pth')